 We build a tiny loan-approval classifier. A specific applicant is rejected. A loan
officer disagrees ("good credit should be enough on its own"). We turn that complaint into a
precise edit and apply it to the compiled classifier.

The seven steps:

| Step | What we do |
|---|---|
| 1 | Compile the classifier into an SDD |
| 2 | Evaluate one applicant, see the rejection | 
| 3 | Find the *reasons* (prime implicants) | 
| 4 | Adjust one reason | 
| 5 | Perform the edit on the SDD |
| 6 | Measure cost (edit vs full recompile) | 
| 7 | Check the edit is sensible (postulates) | 



## Setup




In [1]:

try:
    import pysdd 
    print("PySDD already installed.")
except ImportError:
    import sys, subprocess
    subprocess.check_call([sys.executable, "-m", "pip", "install", "PySDD", "--break-system-packages"])
    print("PySDD installed.")

PySDD installed.


In [2]:
# Core imports:
# SddManager : the "workspace" that owns all SDD nodes and builds new ones
# Vtree      : the "recipe" that decides how variables are grouped/split (controls SDD shape)
from pysdd.sdd import SddManager, Vtree
from itertools import combinations, product

## Step 1: Define and compile the classifier

Our classifier has **three features**, each a yes/no (Boolean) variable:

- `credit`  - is the applicant's credit good?  (variable 1)
- `income`  - is their income stable?           (variable 2)
- `deposit` - did they put down a large deposit? (variable 3)

The classifier **approves a loan** exactly when this logical rule is true:

$$\Delta \;=\; (\textsf{credit} \wedge \textsf{income}) \;\vee\; \textsf{deposit}$$



In [ ]:
# A vtree over 3 variables. var_order lists the variable ids; vtree_type controls the shape
# (The vtree only affects the SDD's internal structure/size, not which function it represents)
vtree = Vtree(var_count=3, var_order=[1, 2, 3], vtree_type="balanced")

# The manager owns every SDD node we will create
mgr = SddManager.from_vtree(vtree)

# Grab one SDD literal per variable. mgr.literal(i) is the SDD for "variable i is True"
credit, income, deposit = [mgr.literal(i) for i in range(1, 4)]

# Human-readable names, so we can print terms nicely later
names = {1: "credit", 2: "income", 3: "deposit"}
lits  = {1: credit, 2: income, 3: deposit}

# Build the classifier Delta = (credit AND income) OR deposit
# In PySDD, & is conjoin (AND), | is disjoin (OR), ~ is negate (NOT)
# Each of these is the fast "Apply" operation running on the compiled diagrams
delta = (credit & income) | deposit

# .size() = number of edges in the SDD (a common size measure)
# .model_count() = how many of the 2^3 = 8 possible applicants get APPROVED
print("Delta compiled.")
print("  SDD size (edges):", delta.size())
print("  Approved applicants (model count):", delta.global_model_count())

Delta compiled.
  SDD size (edges): 4
  Approved applicants (model count): 5


That `model count` of **5** means: of the 8 possible applicants, 5 are approved by $\Delta$.
This number is our ground truth for later : when we edit the classifier, watching how it changes
tells us the edit did what we intended.

## Step 2: Evaluate one applicant and see the rejection

Consider a specific applicant:

- good `credit`  (True)
- **unstable** `income`  (False)
- **small** `deposit`  (False)

We represent the applicant as a conjunction of literals, then ask whether the classifier approves
them. The trick: an applicant is approved iff *(applicant AND $\Delta$)* is still satisfiable.
If the result has **0** models, the applicant is rejected.

In [ ]:
# The applicant: credit=True, income=False, deposit=False
applicant = credit & (~income) & (~deposit)

# Does the classifier approve them?
# (applicant AND delta) describes "this applicant, and approved".
# If that has 0 models, the applicant cannot be approved -> REJECTED
conj = applicant & delta
# If (applicant AND Delta) is the 'false' SDD, the applicant cannot be approved
is_approved = not conj.is_false()
print("Is (applicant AND Delta) satisfiable?", is_approved)
print("Decision:", "APPROVED" if is_approved else "REJECTED")

Is (applicant AND Delta) satisfiable? False
Decision: REJECTED


## Step 3: Find the reasons (prime implicants) 



PySDD has no built-in prime-implicant finder. For this tiny example we can find them by brute force, *using SDD operations
as the engine*. 



In [5]:
def term_to_sdd(term):
    """Turn a term (list of (var_id, sign) pairs) into an SDD conjunction.
    sign=True means the variable is required True; sign=False means required False."""
    t = mgr.true() # start from the 'always true' SDD
    for (v, sign) in term:
        t = t & (lits[v] if sign else ~lits[v])   # AND in each required literal
    return t

def implies_delta(term):
    """Return True iff (term => Delta), i.e. every applicant matching 'term' is approved.
    Test: (term AND NOT Delta) must be unsatisfiable."""
    t = term_to_sdd(term)
    return (t & delta.negate()).is_false()

def is_prime_implicant(term):
    """term is a prime implicant of Delta iff it implies Delta AND no proper
    sub-term already implies Delta (i.e. it is minimal)."""
    if not implies_delta(term):
        return False
    for k in range(len(term)):  # try removing each condition in turn
        smaller = term[:k] + term[k+1:]
        if implies_delta(smaller):  # a smaller term already suffices ->
            return False     # the original was not minimal
    return True

In [ ]:
def find_prime_implicants():
    """Brute-force search over all consistent terms on our 3 variables."""
    vars_ = [1, 2, 3]
    found = []
    for size in range(1, len(vars_) + 1): # consider 1-, 2-, 3-variable terms
        for combo in combinations(vars_, size):    # which variables appear
            for signs in product([True, False], repeat=size):  # True/False for each
                term = list(zip(combo, signs))
                if is_prime_implicant(term):
                    found.append(term)
    return found

def show(term):
    return " AND ".join((("" if s else "NOT ") + names[v]) for v, s in term)

prime_implicants = find_prime_implicants()
print("Reasons the classifier uses to APPROVE (prime implicants of Delta):")
for pi in prime_implicants:
    print("   -", show(pi))

Reasons the classifier uses to APPROVE (prime implicants of Delta):
   - deposit
   - credit AND income


So there are exactly two minimal reasons to approve:

1. **`deposit`** : a large deposit alone is enough.
2. **`credit AND income`** : good credit *together with* stable income.

The complaint is about reason #2: *"`credit AND income` is too strict : good credit
should be enough."* That points us at the precise edit to make.

## Step 4: Adjust one reason 

We **weaken** reason #2 by dropping the `income` condition from it:

$$\textsf{credit} \wedge \textsf{income} \;\longrightarrow\; \textsf{credit}$$

Weakening a reason means *making it easier to satisfy*, so more applicants will now be approved
through it.

In [7]:
# Identify the reason to weaken: credit AND income  (variables 1 and 2, both True).
reason_to_weaken = [(1, True), (2, True)]
print("Original reason:", show(reason_to_weaken))

# Weaken it by dropping the 'income' condition, leaving just 'credit'.
weakened_reason = [(1, True)]
print("Weakened reason:", show(weakened_reason))

Original reason: credit AND income
Weakened reason: credit


## Step 5: Perform the edit on the SDD

A reason-guided edit rebuilds the classifier as the **OR of its reasons, with the chosen reason
weakened**. Concretely, the edited classifier is:

$$\Delta' \;=\; \textsf{deposit} \;\vee\; \textsf{credit}$$

(the untouched reason `deposit`, OR the weakened reason `credit`).

We build $\Delta'$ using the same fast SDD `Apply` operations (`|` and `&`)

In [8]:
def build_from_reasons(reason_list):
    """Rebuild a classifier as the OR (disjoin) of a list of reasons (terms)."""
    result = mgr.false()  # start from 'always false'
    for term in reason_list:
        result = result | term_to_sdd(term) # OR in each reason
    return result

# New reason set: keep 'deposit', replace 'credit AND income' with the weakened 'credit'.
new_reasons = [[(3, True)],          # deposit
               weakened_reason]      # credit (was credit AND income)

delta_edited = build_from_reasons(new_reasons)

print("Edited classifier Delta' built.")
print("  SDD size (edges):", delta_edited.size())
print("  Approved applicants now (model count):", delta_edited.global_model_count())

Edited classifier Delta' built.
  SDD size (edges): 2
  Approved applicants now (model count): 6


The approved count rose from **5** to **6**: weakening the reason let in exactly the kind of
applicant the officer cared about. Let's confirm our specific applicant is now approved.

In [9]:
# Re-evaluate the same applicant (good credit, unstable income, no deposit) on the EDITED classifier
conj_new = applicant & delta_edited
is_approved_now = not conj_new.is_false()
print("Is (applicant AND Delta') satisfiable?", is_approved_now)
print("New decision:", "APPROVED" if is_approved_now else "REJECTED")

Is (applicant AND Delta') satisfiable? True
New decision: APPROVED


## Step 6: Measure the cost

Comparing two ways of producing the edited classifier:

1. **Edit / local-repair** : reuse the existing compiled structure and only change what's needed
   (what we did above: a few Apply operations).
2. **Full recompile** : throw everything away and build $\Delta'$ from scratch.

For a tiny 3-variable example the timing difference is negligible, but the *infrastructure* for
measuring it is the point, and on larger, structured classifiers the local-repair approach is the
one expected to win (this echoes Slater's incremental-update experiment). We time both and also
confirm they produce the **same** function.

In [10]:
import time

# Approach 1: the edit we already performed (reusing compiled reasons)
t0 = time.perf_counter()
edited_via_edit = build_from_reasons(new_reasons)
t_edit = time.perf_counter() - t0

# Approach 2: full recompile of Delta' = credit OR deposit, from the raw formula, in a FRESH manager
t0 = time.perf_counter()
mgr2 = SddManager.from_vtree(Vtree(var_count=3, var_order=[1,2,3], vtree_type="balanced"))
c2, i2, d2 = [mgr2.literal(i) for i in range(1,4)]
edited_via_recompile = c2 | d2
t_recompile = time.perf_counter() - t0

print(f"Edit / local-repair time : {t_edit*1e6:8.1f} microseconds")
print(f"Full recompile time      : {t_recompile*1e6:8.1f} microseconds")
print()
print("Both approve the same number of applicants:",
      edited_via_edit.global_model_count(), "vs", edited_via_recompile.global_model_count())

Edit / local-repair time :    457.9 microseconds
Full recompile time      :  40359.8 microseconds

Both approve the same number of applicants: 6 vs 6


## Step 7: Check the edit is sensible 

Belief-revision theory gives a checklist of **postulates**: conditions any *rational* edit should
satisfy.  Two simple, important ones:

- **Success** : the edit actually achieves its goal (the contested applicant is now approved).
- **Minimal change / preservation** : we only changed decisions we needed to change; previously
  approved applicants stay approved (we *weakened* a reason, so we should only *add* approvals,
  never remove any).

We can check both directly on the SDDs. The second one says: every model of the old $\Delta$ is
still a model of the new $\Delta'$, i.e. $\Delta \Rightarrow \Delta'$.

In [ ]:
# Postulate 1 - Success: the contested applicant is approved by the edited classifier
success = not (applicant & delta_edited).is_false()
print("Success (applicant now approved):", success)

# Postulate 2 - Preservation: every applicant approved before is still approved
# Delta => Delta'  holds iff (Delta AND NOT Delta') is unsatisfiable
# bool(...) because PySDD's is_false() returns an int (1/0); we want a clean True/False
preserved = bool((delta & delta_edited.negate()).is_false())
print("Preservation (no previously-approved applicant got dropped):", preserved)

# Sanity: we should have ADDED exactly the contested kind of applicant (count went 5 -> 6)
print("Approvals before:", delta.global_model_count(), " after:", delta_edited.global_model_count())

Success (applicant now approved): True
Preservation (no previously-approved applicant got dropped): True
Approvals before: 5  after: 6
